In [1]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

# --- CONFIGURATION ---
# 1. Automatically find the '_processed.csv' file in the current directory
input_csv_name = None
for f in os.listdir('.'): # List files in the current directory
    if f.endswith('_processed.csv'):
        input_csv_name = f
        break

if not input_csv_name:
    print("ERRO: Nenhum arquivo '*_processed.csv' encontrado neste diretório.")
    # exit() # Descomente se quiser parar a execução

else:
    print(f"Processando arquivo: {input_csv_name}")

    # 2. Define the TARGET column structure (based on UNSW_NB15_testing-set.csv, excluding 'id')
    target_columns = [
        'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
        'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
        'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean',
        'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
        'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login',
        'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports',
        'attack_cat', 'label'
    ]

    # 3. Define the RENAME mapping (Zeek/Generated Name -> Target Name)
    rename_map = {
        'duration': 'dur',
        'conn_state': 'state',
        'orig_pkts': 'spkts',
        'resp_pkts': 'dpkts',
        'orig_bytes': 'sbytes',
        'resp_bytes': 'dbytes',
        # Simple calculated features already have correct names:
        # 'rate', 'sload', 'dload', 'smean', 'dmean', 'is_sm_ips_ports', 'is_ftp_login'
        # Joined features (keep prefix for now, will handle missing/selection later)
        'http_trans_depth': 'trans_depth',
        'http_response_body_len': 'response_body_len',
        # ct_* features already have correct names
    }

    # 4. Define DEFAULT values for MISSING target columns
    #    Using 0 for numeric, '-' for service/proto/state if needed (already exist mostly)
    default_values = {
        'sttl': 0,
        'dttl': 0,
        'sloss': 0,
        'dloss': 0,
        'sinpkt': 0.0,
        'dinpkt': 0.0,
        'sjit': 0.0,
        'djit': 0.0,
        'swin': 0,
        'stcpb': 0,
        'dtcpb': 0,
        'dwin': 0,
        'tcprtt': 0.0,
        'synack': 0.0,
        'ackdat': 0.0,
        # trans_depth and response_body_len come from http merge, handle NaNs later if needed
        'trans_depth': 0,
        'response_body_len': 0,
        # is_ftp_login is calculated, ct_ftp_cmd uses ftp_command, ct_flw uses http_method - handle NaNs later
    }

    # --- PROCESSING ---
    try:
        # Read the input CSV
        df = pd.read_csv(input_csv_name, low_memory=False)
        print(f"Arquivo {input_csv_name} lido com {len(df)} linhas.")

        # Rename columns based on the map
        df.rename(columns=rename_map, inplace=True)
        print("Colunas renomeadas.")

        # Add missing columns with default values
        missing_cols_added = []
        for col in target_columns:
            if col not in df.columns:
                # Assign default value, using specific defaults if defined, else 0
                df[col] = default_values.get(col, 0)
                missing_cols_added.append(col)
        if missing_cols_added:
            print(f"Colunas ausentes adicionadas com valor padrão: {missing_cols_added}")
        else:
             print("Nenhuma coluna alvo estava ausente.")

        # Select only the target columns IN THE CORRECT ORDER
        # This automatically removes any extra columns (like uid, ts, id.orig_h, etc.)
        df_final = df[target_columns]
        print("Colunas selecionadas e reordenadas para o formato final.")

        # --- FINAL CHECKS and OUTPUT ---
        print("\n--- Estrutura do DataFrame Final ---")
        df_final.info() # Shows column names, count, and types

        print("\n--- Amostra do DataFrame Final ---")
        display(df_final.head())

        # Define output filename
        output_csv_name = input_csv_name.replace('_processed.csv', '_features_processed.csv')

        # Save the final DataFrame
        df_final.to_csv(output_csv_name, index=False)
        print(f"\nDataFrame finalizado salvo como: {output_csv_name}")

    except FileNotFoundError:
        print(f"ERRO: Arquivo de entrada não encontrado: {input_csv_name}")
    except Exception as e:
        print(f"Ocorreu um erro inesperado durante o processamento: {e}")
        import traceback
        traceback.print_exc() # Print detailed error traceback

Processando arquivo: exploits_processed.csv
Arquivo exploits_processed.csv lido com 35189 linhas.
Colunas renomeadas.
Colunas ausentes adicionadas com valor padrão: ['sttl', 'dttl', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat']
Colunas selecionadas e reordenadas para o formato final.

--- Estrutura do DataFrame Final ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35189 entries, 0 to 35188
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   dur                35189 non-null  float64
 1   proto              35189 non-null  object 
 2   service            34985 non-null  object 
 3   state              35189 non-null  object 
 4   spkts              35189 non-null  int64  
 5   dpkts              35189 non-null  int64  
 6   sbytes             35189 non-null  float64
 7   dbytes             35189 non-null  float64
 8   rate       

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000000,udp,dns,S0,1,0,0.0,0.0,0.000000,0,...,1,1,0,0,0,1,1,0,Exploits,1
1,0.000045,tcp,NaN,SH,1,1,0.0,0.0,44444.444444,0,...,1,1,0,0,0,1,1,0,Exploits,1
2,0.000304,tcp,NaN,SH,1,1,0.0,0.0,6578.947368,0,...,1,2,0,0,0,2,1,0,Exploits,1
3,0.000179,tcp,NaN,SH,1,1,0.0,0.0,11173.184358,0,...,1,3,0,0,0,3,1,0,Exploits,1
4,3.080209,udp,NaN,S0,4,0,680.0,0.0,1.298613,0,...,1,1,0,0,0,1,1,0,Exploits,1



DataFrame finalizado salvo como: exploits_features_processed.csv
